In [43]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

### Part 1
- Convert the data from the lab pdf into csv format named pdf_data.data
- read the data into python
- seperate into X and y (last row)

In [44]:
df = pd.read_csv('src/pdf_data.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col

### Part 2
- randomise data and seperate into 60-40 test-train split 

In [45]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(18, 4)
(12, 4)


### Part 3
- change labels from 0 to -1

In [46]:
# y_train = y_train.replace(0, -1)
# y_test = y_test.replace(0, -1)


### Part 4
- change everything to vectors. DF cant be used
- add bias to index 0 in X

In [47]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


print(y_train.shape)
print(X_train.shape)

(18,)
(18, 5)


### Part 5
- now time for the training
- steps?
  - predict
  - check error
  - adjust weights
  - repeat

#### notes:
* on first attemp i used LR of 0.1. this was too high. switching to 0.01

In [48]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-4

fail_safe = 1000
iterations = 0
learning_rate = 0.01

weights = np.array([0.0, 0.0, 0.0, 0.0, 0.0])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1

    print(f"Iterations: {iterations} | current_loss: {current_loss}")


Iterations: 1 | current_loss: 0.6931471805599453
Iterations: 2 | current_loss: 0.6924230026457161
Iterations: 3 | current_loss: 0.6917017817599947
Iterations: 4 | current_loss: 0.690983501726847
Iterations: 5 | current_loss: 0.6902681464571874
Iterations: 6 | current_loss: 0.6895556999485781
Iterations: 7 | current_loss: 0.6888461462850278
Iterations: 8 | current_loss: 0.6881394696367821
Iterations: 9 | current_loss: 0.68743565426011
Iterations: 10 | current_loss: 0.6867346844970852
Iterations: 11 | current_loss: 0.6860365447753631
Iterations: 12 | current_loss: 0.6853412196079526
Iterations: 13 | current_loss: 0.6846486935929839
Iterations: 14 | current_loss: 0.6839589514134709
Iterations: 15 | current_loss: 0.6832719778370703
Iterations: 16 | current_loss: 0.6825877577158355
Iterations: 17 | current_loss: 0.681906275985968
Iterations: 18 | current_loss: 0.681227517667562
Iterations: 19 | current_loss: 0.6805514678643484
Iterations: 20 | current_loss: 0.6798781117634325
Iterations: 21

### Part 6
- verify against test data

In [49]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.6666666666666666


### Part 7
FINAL OUTPUT

model is 
z = w0x1 + w1xBP + w2xChol + w3xAge + w4xPreg | 
y = sigmoid(z)

In [50]:
print(f"""
weights: 
    b: {weights[0]}
    BP: {weights[1]}
    Cholesterol: {weights[2]}
    Age: {weights[3]}
    Pregnant: {weights[4]}

""")


weights: 
    b: -0.23799260957506463
    BP: 0.37234917650898497
    Cholesterol: -0.2615205846893731
    Age: 0.9537671562982688
    Pregnant: 0.7539446731233469




final notes
- some weights can be negative. this just means that it decreases probability of reaction. positive weight increases chanches of reaction